In [9]:
library(tidyverse)
library(oce)
source('../DATA/interpolateData.R')

# =============================================================================
# STEADY-STATE FORCING CALCULATION
# Following Stock et al. (2008): new nutrient flux per area (FN) over
# an idealized well-mixed euphotic zone of depth de.
# =============================================================================

niskin_ds_3 <- readRDS("../DATA/processed/Niskin_cleaned.rds")
CARIACO     <- readRDS("../DATA/processed/CARIACO_EnvData_combined.rds")

# ---- Define depth intervals (adjust per regime as needed) ----
euphotic_lower <- 50

# ---- Extract depth-specific values using interpolateData ----

# NO3 euphotic zone mean (0-50m) — validation target for model N
NO3_euphotic_ds <- interpolateData(
  niskin_ds_3, "NO3_merged",
  depth_from = 0, depth_to = euphotic_lower, noofNA = 10,
  output_type = 'mean', surface_fix = TRUE
)
names(NO3_euphotic_ds)[1] <- "NO3_euphotic"

write.csv(NO3_euphotic_ds, "NO3_euphotic_dynamic.csv", row.names = FALSE)
cat("Saved to NO3_euphotic_dynamic.csv\n")

# Temperature euphotic zone mean (0-50m) — for f-ratio
Temp_euphotic_ds <- interpolateData(
  niskin_ds_3, "Temperature",
  depth_from = 0, depth_to = euphotic_lower, noofNA = 10,
  output_type = 'mean', surface_fix = TRUE
)
names(Temp_euphotic_ds)[1] <- "Temp_euphotic"

# Primary Productivity integrated (0-50m)
PP_ds <- interpolateData(
  niskin_ds_3, "PrimaryProductivity",
  depth_from = 0, depth_to = euphotic_lower, noofNA = 20,
  output_type = 'integrated', surface_fix = TRUE
)
names(PP_ds)[1] <- "PP_integrated"

PP_ds$PP_integrated <- PP_ds$PP_integrated * 12  # hr -> day (12h tropical daylight)

# PON (euphotic zone)
PON_obs <- interpolateData(
  niskin_ds_3, "PN_ug_L",
  depth_from = 0, depth_to = euphotic_lower,
  noofNA = 10, output_type = 'mean', surface_fix = TRUE
)
PON_obs$PON_euphotic <- PON_obs$dat_var / 14.007  # µg/L -> mmol N m-3
PON_obs$dat_var <- NULL

write.csv(PON_obs, "PON_euphotic_dynamic.csv", row.names = FALSE)
cat("Saved to PON_euphotic_dynamic.csv\n")

# =============================================================================
# SEDIMENT TRAP EXPORT FLUX
# Back-calculate export at model boundary from 225m trap
# =============================================================================

mean_trap_225 <- mean(CARIACO$MF_N_mmol_225m, na.rm = TRUE)
sd_trap_225   <- sd(CARIACO$MF_N_mmol_225m, na.rm = TRUE)
n_trap_225    <- sum(!is.na(CARIACO$MF_N_mmol_225m))

cat(sprintf("\n=== Sediment Trap (225m) ===\n"))
cat(sprintf("  Mean flux:    %.4f mmol N m-2 d-1\n", mean_trap_225))
cat(sprintf("  SD:           %.4f mmol N m-2 d-1\n", sd_trap_225))
cat(sprintf("  n months:     %d\n", n_trap_225))

# ---- Martin curve back-calculation to model boundary ----
z_model   <- euphotic_lower  # 50m
z_trap    <- 225
martin_b  <- 0.858           # canonical value

correction_factor <- (z_trap / z_model)^martin_b

export_50m_mean <- mean_trap_225 * correction_factor
export_50m_sd   <- sd_trap_225 * correction_factor
export_volumetric <- export_50m_mean / euphotic_lower  # mmol N m-3 d-1

cat(sprintf("\n=== Export Flux at Model Boundary (%dm) ===\n", z_model))
cat(sprintf("  Martin b (canonical):  %.3f\n", martin_b))
cat(sprintf("  Correction factor:     %.2f\n", correction_factor))
cat(sprintf("  Export flux (mean):    %.4f mmol N m-2 d-1\n", export_50m_mean))
cat(sprintf("  Export flux (SD):      %.4f mmol N m-2 d-1\n", export_50m_sd))
cat(sprintf("  Volumetric equivalent: %.6f mmol N m-3 d-1\n", export_volumetric))

# =============================================================================
# DATA AVAILABILITY
# =============================================================================

cat("\n=== Data availability ===\n")
cat(sprintf("  NO3 euphotic (0-%dm):      %d dates\n", euphotic_lower, nrow(NO3_euphotic_ds)))
cat(sprintf("  Temperature (0-%dm):       %d dates\n", euphotic_lower, nrow(Temp_euphotic_ds)))
cat(sprintf("  PP integrated (0-%dm):     %d dates\n", euphotic_lower, nrow(PP_ds)))
cat(sprintf("  PON euphotic (0-%dm):      %d dates\n", euphotic_lower, nrow(PON_obs)))
cat(sprintf("  Sediment trap (225m):      %d months\n", n_trap_225))

# =============================================================================
# GRAND MEANS
# =============================================================================

mean_euphotic_depth <- mean(CARIACO$euphotic_depth, na.rm = TRUE)
mean_N_surface <- mean(NO3_euphotic_ds$NO3_euphotic, na.rm = TRUE)
mean_temp      <- mean(Temp_euphotic_ds$Temp_euphotic, na.rm = TRUE)
mean_PP        <- mean(PP_ds$PP_integrated, na.rm = TRUE)
mean_PON       <- mean(PON_obs$PON_euphotic, na.rm = TRUE)
mean_export    <- export_50m_mean

cat("\n=== Grand Mean Values ===\n")
cat(sprintf("  Euphotic depth:       %.2f m\n", mean_euphotic_depth))
cat(sprintf("  Temperature (0-%dm):  %.2f °C\n", euphotic_lower, mean_temp))
cat(sprintf("  PP (integrated):      %.2f mg C m-2 d-1\n", mean_PP))
cat(sprintf("  N_surface (0-%dm):    %.4f mmol N m-3\n", euphotic_lower, mean_N_surface))
cat(sprintf("  PON (0-%dm):          %.4f mmol N m-3\n", euphotic_lower, mean_PON))
cat(sprintf("  Export flux (%dm):    %.4f mmol N m-2 d-1 (from 225m trap)\n", z_model, mean_export))

# =============================================================================
# DERIVED FORCING (Stock et al. 2008 style: FN and de)
# =============================================================================

# ---- f-ratio (Laws et al. 2011) ----
f_ratio_calc <- function(T, PP_mgC) {
  (0.5857 - 0.0165 * T) * PP_mgC / (51.7 + PP_mgC)
}

f_ratio <- f_ratio_calc(mean_temp, mean_PP)

# ---- New production → FN (new nutrient flux per area) ----
new_prod_area <- f_ratio * mean_PP                    # mg C m-2 d-1
FN            <- new_prod_area / 12.01 * (16 / 106)   # mmol N m-2 d-1
de            <- euphotic_lower                       # m

cat(sprintf("\n=== Derived Forcing (Stock-style) ===\n"))
cat(sprintf("  f-ratio:         %.4f\n", f_ratio))
cat(sprintf("  New production:  %.4f mg C m-2 d-1\n", new_prod_area))
cat(sprintf("  FN:              %.4f mmol N m-2 d-1\n", FN))
cat(sprintf("  de:              %.1f m\n", de))
cat(sprintf("  FN/de:           %.6f mmol N m-3 d-1\n", FN / de))

# =============================================================================
# CONSISTENCY CHECK: Export vs New Production
# =============================================================================

cat(sprintf("\n=== Consistency Check ===\n"))
cat(sprintf("  New production:  %.4f mmol N m-2 d-1 (FN, f-ratio method)\n", FN))
cat(sprintf("  Export flux:     %.4f mmol N m-2 d-1 (trap-derived)\n", mean_export))
cat(sprintf("  Ratio (exp/FN):  %.2f\n", mean_export / FN))

# =============================================================================
# SUMMARY TABLE
# =============================================================================

forcing_summary <- tibble(
  parameter = c("euphotic_depth", "temperature", "PP_total",
                "f_ratio", "new_production_area", "FN", "de",
                "N_surface", "PON", "export_flux", "martin_b"),
  value = c(mean_euphotic_depth, mean_temp, mean_PP,
            f_ratio, new_prod_area, FN, de,
            mean_N_surface, mean_PON, mean_export, martin_b),
  units = c("m", "deg_C", "mg_C_m-2_d-1",
            "dimensionless", "mg_C_m-2_d-1", "mmol_N_m-2_d-1", "m",
            "mmol_N_m-3", "mmol_N_m-3", "mmol_N_m-2_d-1", "dimensionless"),
  depth_interval = c(
    "from_combined_dataset",
    sprintf("0-%dm", euphotic_lower),
    sprintf("0-%dm_integrated", euphotic_lower),
    "derived_Laws2011", "derived", "derived_Stock2008", "box_depth",
    sprintf("0-%dm", euphotic_lower),
    sprintf("0-%dm", euphotic_lower),
    sprintf("%dm_from_225m_trap", z_model),
    "Martin_1987")
)

print(forcing_summary)
write.csv(forcing_summary, "steady_state_forcing_summary.csv", row.names = FALSE)
cat("\nSaved to steady_state_forcing_summary.csv\n")

Saved to NO3_euphotic_dynamic.csv
Saved to PON_euphotic_dynamic.csv

=== Sediment Trap (225m) ===
  Mean flux:    0.7104 mmol N m-2 d-1
  SD:           0.4274 mmol N m-2 d-1
  n months:     139

=== Export Flux at Model Boundary (50m) ===
  Martin b (canonical):  0.858
  Correction factor:     3.63
  Export flux (mean):    2.5821 mmol N m-2 d-1
  Export flux (SD):      1.5534 mmol N m-2 d-1
  Volumetric equivalent: 0.051641 mmol N m-3 d-1

=== Data availability ===
  NO3 euphotic (0-50m):      211 dates
  Temperature (0-50m):       226 dates
  PP integrated (0-50m):     219 dates
  PON euphotic (0-50m):      214 dates
  Sediment trap (225m):      139 months

=== Grand Mean Values ===
  Euphotic depth:       44.92 m
  Temperature (0-50m):  24.34 °C
  PP (integrated):      1202.97 mg C m-2 d-1
  N_surface (0-50m):    2.0158 mmol N m-3
  PON (0-50m):          9.4094 mmol N m-3
  Export flux (50m):    2.5821 mmol N m-2 d-1 (from 225m trap)

=== Derived Forcing (Stock-style) ===
  f-ratio: 

In [10]:
niskin_ds_3$PN_ug_L

[1]        NA        NA        NA        NA        NA        NA        NA
   [8]        NA        NA        NA        NA        NA        NA        NA
  [15]        NA        NA        NA        NA        NA  58.30000  61.50580
  [22]  51.76150  79.09630  48.60450  26.47600  49.09730  36.14720  19.14710
  [29]  18.58100  18.85430  20.63040  33.25850  25.21710  23.88500  20.70360
  [36]  22.69930  26.64680  27.21770 385.49800 109.59300 304.46000  66.43900
  [43]  72.11870  30.61380  21.00610  21.03050  16.20470  16.73170  12.45730
  [50]  22.38700  30.03810  21.27940  24.84140  22.77740  15.90220  21.55700
  [57]  17.39530 374.78300 370.55700 409.80800 173.70000  29.09630  25.18790
  [64]  20.06440  15.30690  18.41030  20.82070  26.70540  27.90080  27.16400
  [71]  32.19970  28.26680  29.62330  30.61380  27.16400  23.94850 735.22000
  [78] 612.63800 440.84100 354.07400  63.57960  24.49500  24.97800  21.49900
  [85]  30.48700  34.05630  18.82500  63.48690  25.61730  35.77140  25.75390
  [92]  21.52340  23.14820  28.68160  23.14820 359.81300 356.69000 350.73700
  [99] 385.47900 144.96900  77.82770  33.52200  32.98530  43.23220  28.25220
 [106]  12.19870  22.00640  30.64310  32.39970  19.07880  26.00760  23.86060
 [113]  22.59200  25.47090 502.97600 412.90100  47.72130  45.13520  44.69600
 [120]  29.76480  26.49560  28.64250        NA  23.42150  17.41970  40.79240
 [127]  28.10580  35.76660  31.61900  35.27860  26.39800  22.25040  20.49380
 [134] 333.75600 529.13000 150.53200 135.64900 172.24600  33.81480  33.13160
 [141]  38.05990  36.79130  20.98180  16.93180  27.91060  30.69190  34.69310
 [148]  32.59490  28.00820  34.69310  34.05870  27.71540 145.08600 158.16300
 [155]  86.55220  73.40690  65.58990  95.33520  58.01700  29.16950  32.82420
 [162]  25.48550  21.15740  39.29930  27.85690  42.60270  30.71140  26.25160
 [169]  23.33850  28.91580  21.18670  23.27020  63.27430  63.51040  49.43930
 [176] 102.45500  53.96930  32.56310  18.90970  16.83760  11.39300  13.78060
 [183]  15.32380  19.23490  52.19090  31.34580  37.63050  22.95310  29.61840
 [190]  21.53800  16.60000  22.56270  31.15550  27.79350  28.20830  28.27170
 [197]  28.43270  27.12010  12.66220  30.29180        NA  13.95070  19.13270
 [204]  76.83710  33.17560  41.60730  43.21260  32.72670  34.49300  44.26660
 [211]  32.76080  44.04220 152.36200 122.36300  85.59580 187.62100 148.77000
 [218] 122.36300  42.95890  24.86580  36.12760  38.37220  34.64920  54.50860
 [225]  45.20350  40.87540  59.41740  39.36760  28.52050  52.48850  53.19120
 [232]  53.19120 155.41100 127.11000 162.50100 108.31000  67.21480  39.36760
 [239]  22.55290  21.55750  31.57020  24.31930  22.26510  46.03790  47.03330
 [246]  31.53610  34.32710  39.49450  31.12130  29.57940  43.34440  36.41550
 [253]  62.91600  69.01040  74.81700  69.90830  92.39780  45.97440  35.77140
 [260]  44.14460  30.57480  39.55790  27.91060  33.14140  51.26870  44.53010
 [267]  54.28420  38.82110  36.57660  40.68020  39.84580  40.16790 335.31800
 [274] 310.90100 167.35600  89.54330 103.98700  76.28090  78.04470  96.48350
 [281]  71.43550  77.68620  66.94150  77.30070 133.82000  77.08110  71.88450
 [288]  69.87410  59.76870  54.31830  70.99150  57.89010 169.04900 180.53600
 [295] 232.15600 266.03400 114.05300  88.75280  58.15850  64.09190  33.91240
 [302]  23.99240  27.62760  48.61910  41.60240  45.65240  33.56100  51.74690
 [309]  49.38520  35.66900  49.82920  48.17510 456.86400 578.73100 682.28700
 [316] 100.16100 437.42600  56.36500  45.96400  52.30100  34.74680  33.02430
 [323]  47.68710  63.37460  53.42540  42.49050  38.60150  40.73880  51.32230
 [330]  37.90380  44.02260  49.79020 340.95800 309.46300 195.47200 103.60300
 [337]  75.35860  39.90920  27.63740  39.97270  32.00450  24.80240  19.54230
 [344]  51.22470  40.26060  36.24480  32.99500  61.87180  30.12590  29.48670
 [351]  41.98300  65.66800 228.59400 259.99300 173.67000 102.99600  73.79720
 [358]  60.82270  45.77440  35.51280        NA  39.43110        NA        NA

In [12]:
str(niskin_ds_3)

'data.frame':	4394 obs. of  102 variables:
 $ Cruise_number              : int  1 1 1 1 1 1 1 1 1 1 ...
 $ Cruise_ID_1                : chr  "93HG_001" "93HG_001" "93HG_001" "93HG_001" ...
 $ Cruise_ID_2                : chr  "CAR-001" "CAR-001" "CAR-001" "CAR-001" ...
 $ Leg                        : int  2 2 2 2 2 2 2 2 2 2 ...
 $ Day                        : int  8 8 8 8 8 8 8 8 8 8 ...
 $ Month                      : int  11 11 11 11 11 11 11 11 11 11 ...
 $ Year                       : int  1995 1995 1995 1995 1995 1995 1995 1995 1995 1995 ...
 $ Latitude                   : num  10.5 10.5 10.5 10.5 10.5 10.5 10.5 10.5 10.5 10.5 ...
 $ Longitude                  : num  -64.7 -64.7 -64.7 -64.7 -64.7 ...
 $ Hydro_cast_no              : int  1 1 1 1 1 1 1 1 3 3 ...
 $ Depth_target               : int  1 7 15 25 35 55 75 100 150 200 ...
 $ Depth_real                 : num  1.5 6.5 15 25 35 ...
 $ O2_ml_L                    : num  4.85 4.41 4.38 4.37 4.27 3.95 3.87 3.63 1.81 0.45 ...
 $

In [8]:
library(tidyverse)
library(oce)
source('../DATA/interpolateData.R')

# =============================================================================
# STEADY-STATE FORCING CALCULATION
# Following Stock et al. (2008) chemostat framework
#
# Depth intervals are defined here and can be adjusted per regime.
# =============================================================================

niskin_ds_3 <- readRDS("../DATA/processed/Niskin_cleaned.rds")
CARIACO     <- readRDS("../DATA/processed/CARIACO_EnvData_combined.rds")

# ---- Define depth intervals (adjust per regime as needed) ----
euphotic_lower <- 50
N0_upper       <- 50
N0_lower       <- 70

# ---- Extract depth-specific values using interpolateData ----

# NO3 euphotic zone mean (0-50m) — model state variable N
NO3_euphotic_ds <- interpolateData(
  niskin_ds_3, "NO3_merged",
  depth_from = 0, depth_to = euphotic_lower, noofNA = 10,
  output_type = 'mean', surface_fix = TRUE
)
names(NO3_euphotic_ds)[1] <- "NO3_euphotic"

write.csv(NO3_euphotic_ds, "NO3_euphotic_dynamic.csv", row.names = FALSE)
cat("Saved to NO3_euphotic_dynamic.csv\n")

# NO3 sub-euphotic mean (50-70m) — model forcing N0
NO3_subeuphotic_ds <- interpolateData(
  niskin_ds_3, "NO3_merged",
  depth_from = N0_upper, depth_to = N0_lower, noofNA = 10,
  output_type = 'mean', surface_fix = FALSE
)
names(NO3_subeuphotic_ds)[1] <- "NO3_subeuphotic"

# Temperature euphotic zone mean (0-50m) — for f-ratio
Temp_euphotic_ds <- interpolateData(
  niskin_ds_3, "Temperature",
  depth_from = 0, depth_to = euphotic_lower, noofNA = 10,
  output_type = 'mean', surface_fix = TRUE
)
names(Temp_euphotic_ds)[1] <- "Temp_euphotic"

# Primary Productivity integrated (0-50m)
PP_ds <- interpolateData(
  niskin_ds_3, "PrimaryProductivity",
  depth_from = 0, depth_to = euphotic_lower, noofNA = 20,
  output_type = 'integrated', surface_fix = TRUE
)
names(PP_ds)[1] <- "PP_integrated"

PP_ds$PP_integrated <- PP_ds$PP_integrated * 12  # hr -> day (12h tropical daylight)

# PON (euphotic zone)
PON_obs <- interpolateData(
  niskin_ds_3, "PN_ug_L",
  depth_from = 0, depth_to = euphotic_lower,
  noofNA = 10, output_type = 'mean', surface_fix = TRUE
)
PON_obs$PON_euphotic <- PON_obs$dat_var / 14.007  # µg/L -> mmol N m-3
PON_obs$dat_var <- NULL

write.csv(PON_obs, "PON_euphotic_dynamic.csv", row.names = FALSE)
cat("Saved to PON_euphotic_dynamic.csv\n")

# =============================================================================
# SEDIMENT TRAP EXPORT FLUX
# Back-calculate export at model boundary from 225m trap
# =============================================================================

# ---- Extract 225m trap data from combined dataset ----
mean_trap_225 <- mean(CARIACO$MF_N_mmol_225m, na.rm = TRUE)
sd_trap_225   <- sd(CARIACO$MF_N_mmol_225m, na.rm = TRUE)
n_trap_225    <- sum(!is.na(CARIACO$MF_N_mmol_225m))

cat(sprintf("\n=== Sediment Trap (225m) ===\n"))
cat(sprintf("  Mean flux:    %.4f mmol N m-2 d-1\n", mean_trap_225))
cat(sprintf("  SD:           %.4f mmol N m-2 d-1\n", sd_trap_225))
cat(sprintf("  n months:     %d\n", n_trap_225))

# ---- Martin curve back-calculation to model boundary ----
z_model   <- euphotic_lower  # 50m
z_trap    <- 225
martin_b  <- 0.858           # canonical value

correction_factor <- (z_trap / z_model)^martin_b

export_50m_mean <- mean_trap_225 * correction_factor
export_50m_sd   <- sd_trap_225 * correction_factor
export_volumetric <- export_50m_mean / euphotic_lower  # mmol N m-3 d-1

cat(sprintf("\n=== Export Flux at Model Boundary (%dm) ===\n", z_model))
cat(sprintf("  Martin b (canonical):  %.3f\n", martin_b))
cat(sprintf("  Correction factor:     %.2f\n", correction_factor))
cat(sprintf("  Export flux (mean):    %.4f mmol N m-2 d-1\n", export_50m_mean))
cat(sprintf("  Export flux (SD):      %.4f mmol N m-2 d-1\n", export_50m_sd))
cat(sprintf("  Volumetric equivalent: %.6f mmol N m-3 d-1\n", export_volumetric))

# =============================================================================
# DATA AVAILABILITY
# =============================================================================

cat("\n=== Data availability ===\n")
cat(sprintf("  NO3 euphotic (0-%dm):      %d dates\n", euphotic_lower, nrow(NO3_euphotic_ds)))
cat(sprintf("  NO3 sub-euphotic (%d-%dm): %d dates\n", N0_upper, N0_lower, nrow(NO3_subeuphotic_ds)))
cat(sprintf("  Temperature (0-%dm):       %d dates\n", euphotic_lower, nrow(Temp_euphotic_ds)))
cat(sprintf("  PP integrated (0-%dm):     %d dates\n", euphotic_lower, nrow(PP_ds)))
cat(sprintf("  PON euphotic (0-%dm):      %d dates\n", euphotic_lower, nrow(PON_obs)))
cat(sprintf("  Sediment trap (225m):      %d months\n", n_trap_225))

# =============================================================================
# GRAND MEANS
# =============================================================================

mean_euphotic_depth <- mean(CARIACO$euphotic_depth, na.rm = TRUE)
mean_N0        <- mean(NO3_subeuphotic_ds$NO3_subeuphotic, na.rm = TRUE)
mean_N_surface <- mean(NO3_euphotic_ds$NO3_euphotic, na.rm = TRUE)
mean_temp      <- mean(Temp_euphotic_ds$Temp_euphotic, na.rm = TRUE)
mean_PP        <- mean(PP_ds$PP_integrated, na.rm = TRUE)
mean_PON       <- mean(PON_obs$PON_euphotic, na.rm = TRUE)
mean_export    <- export_50m_mean

cat("\n=== Grand Mean Values ===\n")
cat(sprintf("  Euphotic depth:       %.2f m\n", mean_euphotic_depth))
cat(sprintf("  Temperature (0-%dm):  %.2f °C\n", euphotic_lower, mean_temp))
cat(sprintf("  PP (integrated):      %.2f mg C m-2 d-1\n", mean_PP))
cat(sprintf("  N0 (%d-%dm NO3):      %.4f mmol N m-3\n", N0_upper, N0_lower, mean_N0))
cat(sprintf("  N_surface (0-%dm):    %.4f mmol N m-3\n", euphotic_lower, mean_N_surface))
cat(sprintf("  PON (0-%dm):          %.4f mmol N m-3\n", euphotic_lower, mean_PON))
cat(sprintf("  Export flux (%dm):    %.4f mmol N m-2 d-1 (from 225m trap)\n", z_model, mean_export))

# =============================================================================
# DERIVED FORCING
# =============================================================================

# ---- f-ratio (Laws et al. 2011) ----
f_ratio_calc <- function(T, PP_mgC) {
  (0.5857 - 0.0165 * T) * PP_mgC / (51.7 + PP_mgC)
}

f_ratio <- f_ratio_calc(mean_temp, mean_PP)

# ---- New production ----
new_prod_area     <- f_ratio * mean_PP                                        # mg C m-2 d-1
new_nutrient_flux <- new_prod_area / mean_euphotic_depth / 12.01 * (16 / 106) # mmol N m-3 d-1

# ---- Dilution rate ----
delta_N       <- mean_N0 - mean_N_surface
dilution_rate <- new_nutrient_flux / delta_N

cat(sprintf("\n=== Derived Forcing ===\n"))
cat(sprintf("  f-ratio:              %.4f\n", f_ratio))
cat(sprintf("  New production:       %.4f mg C m-2 d-1\n", new_prod_area))
cat(sprintf("  Nutrient flux:        %.6f mmol N m-3 d-1\n", new_nutrient_flux))
cat(sprintf("  N0 - N_surface:       %.4f mmol N m-3\n", delta_N))
cat(sprintf("  Dilution rate (d):    %.6f d-1\n", dilution_rate))
cat(sprintf("  Residence time (1/d): %.1f days\n", 1 / dilution_rate))

# =============================================================================
# CONSISTENCY CHECK: Export vs New Production
# =============================================================================

new_prod_N <- new_prod_area / 12.01 * (16/106)  # mg C m-2 d-1 -> mmol N m-2 d-1

cat(sprintf("\n=== Consistency Check ===\n"))
cat(sprintf("  New production:    %.4f mmol N m-2 d-1 (f-ratio method)\n", new_prod_N))
cat(sprintf("  Export flux:       %.4f mmol N m-2 d-1 (trap-derived)\n", mean_export))
cat(sprintf("  Ratio (exp/new):   %.2f\n", mean_export / new_prod_N))

# =============================================================================
# SUMMARY TABLE
# =============================================================================

forcing_summary <- tibble(
  parameter = c("euphotic_depth", "temperature", "PP_total",
                "f_ratio", "new_production_area", "new_nutrient_flux",
                "N0", "N_surface", "N0_minus_N", "dilution_rate", "residence_time",
                "PON", "export_flux", "martin_b"),
  value = c(mean_euphotic_depth, mean_temp, mean_PP,
            f_ratio, new_prod_area, new_nutrient_flux,
            mean_N0, mean_N_surface, delta_N, dilution_rate, 1 / dilution_rate,
            mean_PON, mean_export, martin_b),
  units = c("m", "deg_C", "mg_C_m-2_d-1",
            "dimensionless", "mg_C_m-2_d-1", "mmol_N_m-3_d-1",
            "mmol_N_m-3", "mmol_N_m-3", "mmol_N_m-3", "d-1", "days",
            "mmol_N_m-3", "mmol_N_m-2_d-1", "dimensionless"),
  depth_interval = c(
    "from_combined_dataset",
    sprintf("0-%dm", euphotic_lower),
    sprintf("0-%dm_integrated", euphotic_lower),
    "derived_Laws2011", "derived", "derived",
    sprintf("%d-%dm", N0_upper, N0_lower),
    sprintf("0-%dm", euphotic_lower),
    "derived", "derived", "derived",
    sprintf("0-%dm", euphotic_lower),
    sprintf("%dm_from_225m_trap", z_model),
    "Martin_1987")
)

print(forcing_summary)
write.csv(forcing_summary, "steady_state_forcing_summary.csv", row.names = FALSE)
cat("\nSaved to steady_state_forcing_summary.csv\n")

Saved to NO3_euphotic_dynamic.csv
Saved to PON_euphotic_dynamic.csv

=== Sediment Trap (225m) ===
  Mean flux:    0.7104 mmol N m-2 d-1
  SD:           0.4274 mmol N m-2 d-1
  n months:     139

=== Export Flux at Model Boundary (50m) ===
  Martin b (canonical):  0.858
  Correction factor:     3.63
  Export flux (mean):    2.5821 mmol N m-2 d-1
  Export flux (SD):      1.5534 mmol N m-2 d-1
  Volumetric equivalent: 0.051641 mmol N m-3 d-1

=== Data availability ===
  NO3 euphotic (0-50m):      211 dates
  NO3 sub-euphotic (50-70m): 219 dates
  Temperature (0-50m):       226 dates
  PP integrated (0-50m):     219 dates
  PON euphotic (0-50m):      214 dates
  Sediment trap (225m):      139 months

=== Grand Mean Values ===
  Euphotic depth:       44.92 m
  Temperature (0-50m):  24.34 °C
  PP (integrated):      1202.97 mg C m-2 d-1
  N0 (50-70m NO3):      5.5564 mmol N m-3
  N_surface (0-50m):    2.0158 mmol N m-3
  PON (0-50m):          9.4094 mmol N m-3
  Export flux (50m):    2.5821 m